# SWIN AI — обучение математике и программированию

Этот ноутбук дообучает предварительно обученную `Qwen/Qwen3-1.7B` через 4-bit QLoRA. В выборке — сложная математика и задачи программирования; обычные приветствия в этот этап не добавляются.

**Важно:** нужен GPU runtime. Выходной каталог находится на Google Drive, чтобы чекпоинты переживали сброс сессии Colab. Обученный адаптер пока не подключён к Android API автоматически.

In [ ]:
from pathlib import Path
import os

REPO_DIR = Path('/content/Kkkkkk')
if not (REPO_DIR / '.git').exists():
    !git clone https://github.com/celovekajca182-debug/Kkkkkk.git $REPO_DIR
%cd /content/Kkkkkk
!git pull --ff-only
!pip -q install -r server/requirements-qlora.txt
!pip -q install datasets==3.3.2

In [ ]:
import torch
assert torch.cuda.is_available(), 'В Colab включи Runtime → Change runtime type → T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/SWIN_AI/qlora-qwen3-1.7b-math-code'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print('Чекпоинты будут сохраняться в:', OUTPUT_DIR)

## 1. Подготовка Python-задач

Скачивается ограниченная выборка MBPP sanitized. Подготовщик проверяет синтаксис через AST, но не исполняет код из датасета.

In [ ]:
%cd /content/Kkkkkk/server
!python prepare_coding_data.py --max-train 700 --max-validation 160
!ls -lh data/mbpp_python_*

## 2. Настройки обучения

Начни с этих параметров для T4. Если Colab выдаёт CUDA out-of-memory, снизь `MAX_SEQ_LENGTH` до 1024. Для публикации адаптера создай репозиторий модели на Hugging Face и включи `PUBLISH_TO_HUB`.

In [ ]:
from pathlib import Path

# Укажи папку своей готовой базовой модели на Google Drive.
# Она должна содержать config.json, tokenizer_config.json и веса .safetensors/.bin.
# Например: /content/drive/MyDrive/SWIN_AI/my-base-model
DRIVE_BASE_MODEL_PATH = ''

# Если путь выше оставить пустым, будет использована открытая база Qwen3-1.7B.
BASE_MODEL = DRIVE_BASE_MODEL_PATH.strip() or 'Qwen/Qwen3-1.7B'
if DRIVE_BASE_MODEL_PATH.strip():
    base_dir = Path(DRIVE_BASE_MODEL_PATH).expanduser()
    if not base_dir.is_dir():
        raise FileNotFoundError(f'Папка модели не найдена: {base_dir}')
    if not (base_dir / 'config.json').is_file():
        raise ValueError(
            'Это не папка модели Hugging Face: нет config.json. '
            'Если у тебя .pt/.pth/.ckpt, сначала нужно экспортировать веса '
            'в совместимый формат и знать исходную архитектуру.'
        )

MAX_NUMINA = 6000
MAX_HENDRYCKS_PER_CONFIG = 250
MAX_HENDRYCKS_VALIDATION_PER_CONFIG = 100
MAX_SEQ_LENGTH = 1536
EPOCHS = 2
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 8

# Публикация выключена, пока явно не создашь свой репозиторий модели.
PUBLISH_TO_HUB = False
HF_REPO_ID = 'YOUR_USERNAME/swin-ai-math-code'
print('Базовая модель:', BASE_MODEL)
print('Результаты:', OUTPUT_DIR)
print('Публикация в Hub:', PUBLISH_TO_HUB)

In [ ]:
if PUBLISH_TO_HUB:
    from huggingface_hub import notebook_login
    notebook_login()

## 3. Запуск QLoRA

В обучении используются NuminaMath-CoT, семь разделов Hendrycks MATH, MBPP и локальные проверенные примеры SWIN. Validation-наборы не используются для обновления весов.

In [ ]:
cmd = [
    'python train_swin_qlora.py',
    f'--base-model {BASE_MODEL}',
    f'--max-numina {MAX_NUMINA}',
    f'--max-opencode-train {MAX_OPENCODE_TRAIN}',
    f'--max-opencode-validation {MAX_OPENCODE_VALIDATION}',
    f'--max-hendrycks-per-config {MAX_HENDRYCKS_PER_CONFIG}',
    f'--max-hendrycks-validation-per-config {MAX_HENDRYCKS_VALIDATION_PER_CONFIG}',
    f'--max-seq-length {MAX_SEQ_LENGTH}',
    f'--epochs {EPOCHS}',
    f'--batch-size {BATCH_SIZE}',
    f'--gradient-accumulation {GRADIENT_ACCUMULATION}',
    f'--output-dir {OUTPUT_DIR}',
]
if PUBLISH_TO_HUB:
    if 'YOUR_USERNAME' in HF_REPO_ID:
        raise ValueError('Сначала укажи свой HF_REPO_ID.')
    cmd += ['--push-to-hub', '--hub-model-id', HF_REPO_ID]
command = ' '.join(cmd)
print(command)
!{command}

## 4. Что делать после обучения

- Проверь `swin_training_manifest.json` и validation loss.
- Проверь модель на новых, не попавших в обучение задачах математики и кода.
- Для кода запускай тесты только в отдельной песочнице.
- Чтобы SWIN отвечала в Android-приложении, inference-сервис должен загрузить базовую Qwen3 и этот PEFT-адаптер. Само появление файлов на Hugging Face не меняет адрес API в приложении.

Низкий validation loss не доказывает математическую точность или успешное выполнение кода.